---
## 1. Setup

**O que fazemos:** Importar bibliotecas e carregar o Motor de Inferência das aulas anteriores.

✅ Execute esta célula uma única vez.

In [21]:
import requests
import joblib
import numpy as np
import pandas as pd
import json
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Set, Tuple

print('Ambiente pronto!')

Ambiente pronto!


In [23]:
# ============================================================
# MOTOR DE INFERÊNCIA
# ============================================================

@dataclass(frozen=True)
class Rule:
    nome: str
    condicoes: tuple
    acao: str
    prioridade: int
    justificativa: str

    def __post_init__(self):
        object.__setattr__(self, 'condicoes', tuple(self.condicoes))

    def match(self, fatos: Set[str]) -> bool:
        return all(c in fatos for c in self.condicoes)


class WorkingMemory:
    def __init__(self, fatos_iniciais: List[str]):
        self.fatos: Set[str] = set(fatos_iniciais)
        self.historico: List[Tuple[str, str]] = []

    def adicionar(self, fato: str, origem: str = 'initial'):
        if fato not in self.fatos:
            self.fatos.add(fato)
            self.historico.append((fato, origem))


class InferenceEngine:
    def __init__(self, regras: List[Rule]):
        self.regras = sorted(regras, key=lambda r: r.prioridade, reverse=True)
        self.fired_log: List[Dict] = []

    def executar(self, wm: WorkingMemory, verbose: bool = True) -> Dict:
        regras_disparadas = set()
        ciclo = 0
        while True:
            ciclo += 1
            agenda = [r for r in self.regras
                      if r.match(wm.fatos) and r.nome not in regras_disparadas]
            if not agenda:
                if verbose:
                    print(f'  Ciclo {ciclo}: agenda vazia — motor para.')
                break
            selecionada = agenda[0]
            if verbose:
                print(f"  Ciclo {ciclo}: FIRE '{selecionada.nome}' → '{selecionada.acao}'")
            wm.adicionar(selecionada.acao, origem=selecionada.nome)
            regras_disparadas.add(selecionada.nome)
            self.fired_log.append({
                'ciclo': ciclo,
                'regra': selecionada.nome,
                'acao': selecionada.acao,
                'justificativa': selecionada.justificativa
            })
        decisao = next((f for f in wm.fatos if f.startswith('decisao(')), 'decisao(INDEFINIDA)')
        return {'wm_final': wm.fatos, 'decisao': decisao, 'ciclos': ciclo - 1}

    def por_que(self, fato_alvo: str) -> Dict:
        suporte = [e for e in self.fired_log if e['acao'] == fato_alvo]
        if not suporte:
            return {'fato': fato_alvo, 'origem': 'fato inicial (sensor / grounding)'}
        entry = suporte[0]
        regra = next(r for r in self.regras if r.nome == entry['regra'])
        return {
            'fato': fato_alvo,
            'regra': entry['regra'],
            'justificativa': entry['justificativa'],
            'condicoes_necessarias': [self.por_que(c) for c in regra.condicoes]
        }

print('Motor de Inferência carregado')

Motor de Inferência carregado


---
## 2. Módulo de Visão — Azure Custom Vision

### O que o Azure retorna?

Quando enviamos uma imagem para a API, recebemos uma lista com a probabilidade de cada classe treinada:

```json
{
  "predictions": [
    { "tagName": "leg_broken",   "probability": 0.91 },
    { "tagName": "no_defected",  "probability": 0.05 },
    { "tagName": "no_support",   "probability": 0.03 },
    { "tagName": "bed_no_stick", "probability": 0.01 },
    { "tagName": "no_bottom",    "probability": 0.00 }
  ]
}
```

Nossa função seleciona a **tag com maior probabilidade** — essa é a classe prevista.

In [ ]:
# ============================================================
# CONFIGURAÇÃO — Azure Custom Vision
# ============================================================

AZURE_ENDPOINT = 'https://custom3dnunes-prediction.cognitiveservices.azure.com/customvision/v3.0/Prediction/e8d10b1f-7a63-4a9a-b326-85145106ff5b/classify/iterations/Aula3/image'
AZURE_KEY      = os.environ.get('AZURE_CUSTOM_VISION_KEY', 'COLOQUE_SUA_CHAVE_AQUI')


def chamar_azure(imagem_path: str) -> dict:
    """
    Envia uma imagem para o Azure Custom Vision e retorna a predição principal.

    Headers usados (conforme documentação da API):
      - Prediction-Key : chave de autenticação do projeto
      - Content-Type   : application/octet-stream (arquivo binário)

    Retorna: { 'tag': str, 'probabilidade': float, 'todas': list }
    """
    with open(imagem_path, 'rb') as f:
        imagem_bytes = f.read()

    headers = {
        'Prediction-Key': AZURE_KEY,
        'Content-Type':   'application/octet-stream'
    }

    response = requests.post(AZURE_ENDPOINT, headers=headers, data=imagem_bytes)
    response.raise_for_status()

    predictions = response.json()['predictions']
    melhor = max(predictions, key=lambda p: p['probability'])

    return {
        'tag':           melhor['tagName'],
        'probabilidade': melhor['probability'],
        'todas':         predictions
    }

print('Módulo Azure configurado — função chamar_azure() pronta.')

Módulo Azure configurado — função chamar_azure() pronta.


---
## 3. Módulo de Vibração — SVM (.joblib) + CSV de features

### O que é o `vibration_features.csv`?

É o arquivo que vocês geraram na **Aula 5 de RP** (Feature Engineering).
Vocês pegaram as leituras brutas do sensor ADXL345 e calcularam **27 features** por janela de tempo:

| Grupo | Features | Eixos |
|---|---|---|
| Temporais | RMS, média, desvio padrão, máx, mín, curtose, assimetria | X, Y, Z → 21 features |
| Espectrais (FFT) | média FFT, máximo FFT | X, Y, Z → 6 features |

**Resultado:** 32 janelas × 27 features + 1 coluna `label` (`yes`/`no`)

---

### Como funciona na vida real × como funciona hoje

| | Fábrica real | Aula de hoje |
|---|---|---|
| **Fonte** | Sensor ADXL345 ao vivo durante a impressão | Uma linha do `vibration_features.csv` |
| **Features** | Calculadas em tempo real | Já extraídas e salvas na aula de RP |
| **Resultado no SVM** | Idêntico | Idêntico |

> 💡 O mecanismo é exatamente o mesmo. O que muda é só a origem dos 27 números.
> Hoje sorteamos uma linha do CSV. Numa fábrica real, esses números viriam do sensor ao vivo.

---

### ⬆️ Faça o upload do arquivo agora

Execute a célula abaixo e selecione o arquivo `vibration_features.csv` quando solicitado.


In [25]:
# ============================================================
# UPLOAD A — CSV de features de vibração
# (Adaptado para VS Code — usa caminho local)
# ============================================================

CSV_VIBRACAO_PATH = r'vibration_features.csv'

df_vibracao  = pd.read_csv(CSV_VIBRACAO_PATH)
feature_cols = [c for c in df_vibracao.columns if c != 'label']

# --- Correção de formato numérico brasileiro ---
for col in feature_cols:
    if df_vibracao[col].dtype == object:
        df_vibracao[col] = (
            df_vibracao[col]
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .astype(float)
        )

print(f'✅ CSV carregado!')
print(f'   Caminho             : {CSV_VIBRACAO_PATH}')
print(f'   Janelas de vibração : {len(df_vibracao)}')
print(f'   Features por janela : {len(feature_cols)}')
print(f'   Distribuição label  : {df_vibracao["label"].value_counts().to_dict()}')
print(f'   Tipos das colunas   : {df_vibracao[feature_cols].dtypes.unique().tolist()}')
print()
print('💡 Cada linha = uma janela de vibração da impressora.')
print('   Na fábrica real, essa linha viria do sensor ADXL345 ao vivo.')
df_vibracao[feature_cols].head(3)

✅ CSV carregado!
   Caminho             : vibration_features.csv
   Janelas de vibração : 32
   Features por janela : 27
   Distribuição label  : {'yes': 25, 'no': 7}
   Tipos das colunas   : [<StringDtype(storage='python', na_value=nan)>, dtype('float64')]

💡 Cada linha = uma janela de vibração da impressora.
   Na fábrica real, essa linha viria do sensor ADXL345 ao vivo.


,X_rms,X_mean,X_std,X_max,X_min,X_kurt,X_skew,Y_rms,Y_mean,Y_std,...,Z_max,Z_min,Z_kurt,Z_skew,X_fft_mean,X_fft_max,Y_fft_mean,Y_fft_max,Z_fft_mean,Z_fft_max
0,25.722.072.233.783.900,-8.687.500.000.000.000,24.210.583.300.490.700,0.35,-0.78,9.449.604.384.115.940,-9.513.177.424.547.450,9.065.921.767.807.170,8.709.375,2.517.483.785.325.140,...,-5.73,-10.36,2.795.515.600.691.530,19.746.311.980.580.100,18.396.318.630.773.100,55.600.000.000.000.000,35.310.299.515.952.300,55.739.999.999.999.900,2.243.876.692.555.570,5.141.299.999.999.990
1,23.596.543.708.772.200,-7.921.875,2.222.702.794.987.160,0.39,-0.67,18.618.763.425.788.400,-4.431.290.356.202.970,9.250.591.197.323.550,0.89,25.225.854.792.256.200,...,-5.73,-10.40,3.843.611.909.610.490,10.441.002.634.979.200,16.457.349.440.992.100,05.07,35.377.283.533.155.900,56.959.999.999.999.900,21.871.855.807.383.900,51.016.999.999.999.900
2,21.664.342.824.096.900,-9.812.499.999.999.990,19.314.724.790.946.400,0.39,-0.63,7.671.534.925.187.790,-2.715.419.492.040.160,1.025.102.891.177.270,100.609.375,19.649.759.212.503.700,...,11.77,-10.40,-19.714.536.380.415.500,11.461.050.146.393.500,15.247.494.635.734.600,6.28,33.567.932.674.123.800,64.39,39.778.073.625.645,3.962.990.363.569.310


In [33]:
# ============================================================
# UPLOAD B — Modelos treinados na aula de RP
# (Adaptado para VS Code — usa caminhos locais)
# ============================================================

SVM_MODEL_PATH = r'svm_model.joblib'
SCALER_PATH    = r'scaler.joblib'

svm_vibracao    = joblib.load(SVM_MODEL_PATH)
scaler_vibracao = joblib.load(SCALER_PATH)

print(f'✅ SVM carregado!')
print(f'   Caminho         : {SVM_MODEL_PATH}')
print(f'   Kernel          : {svm_vibracao.kernel}')
print(f'   Classes         : {list(svm_vibracao.classes_)}')
print(f'   Vetores suporte : {svm_vibracao.n_support_}')
print(f'\n✅ Scaler carregado!')
print(f'   Caminho         : {SCALER_PATH}')
print(f'   Features        : {scaler_vibracao.n_features_in_}')
print()
print('💡 svm_model.joblib  = o cérebro do classificador (o que vocês treinaram)')
print('   scaler.joblib     = a régua de normalização (obrigatória — sem ela, resultado errado)')

✅ SVM carregado!
   Caminho         : svm_model.joblib
   Kernel          : rbf
   Classes         : [np.str_('defected'), np.str_('no_defected')]
   Vetores suporte : [175 181]

✅ Scaler carregado!
   Caminho         : scaler.joblib
   Features        : 7

💡 svm_model.joblib  = o cérebro do classificador (o que vocês treinaram)
   scaler.joblib     = a régua de normalização (obrigatória — sem ela, resultado errado)


In [ ]:
# ============================================================
# UPLOAD C — Contexto operacional das peças (simula MES)
# (Adaptado para VS Code — usa caminho local)
# ============================================================

CONTEXTO_PATH = r'contexto_pecas - contexto_pecas.csv'

df_contexto = pd.read_csv(CONTEXTO_PATH)
df_contexto = df_contexto.set_index('id')

print(f'✅ Contexto carregado — {len(df_contexto)} peças!')
print(f'   Caminho : {CONTEXTO_PATH}')
print()
print(df_contexto.to_string())
print()
print('💡 Esses dados representam o que o MES saberia sobre cada peça:')
print('   criticidade → qual o impacto se essa peça falhar')
print('   prazo_horas → quanto tempo falta para a entrega')
print()
print(f'   Distribuição criticidade : {df_contexto["criticidade"].value_counts().to_dict()}')
print(f'   Prazo curto (<=24h)      : {(df_contexto["prazo_horas"] <= 24).sum()} peças')
print(f'   Prazo longo (>24h)       : {(df_contexto["prazo_horas"] > 24).sum()} peças')

✅ Contexto carregado — 17 peças!
   Caminho : c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\contexto_pecas - contexto_pecas.csv

                 criticidade  prazo_horas
id                                       
scratch_4_3             alta           45
scratch_4_4            media           20
scratch_3_4            baixa           40
scratch_3_5            media           30
bed_not_stick_1         alta           52
bed_not_stick_0         alta           38
bed_not_stick_10       baixa           24
no_support_34          media           63
no_support_234         baixa           22
no_support_235         media           15
no_support_236          alta           27
leg_broken_118         baixa           68
leg_broken_119          alta           55
leg_broken_120         media           29
no_bottom_4            baixa           33
no_bottom_6            baixa           24
no_bottom_5            media           18

💡 Esses dados representam o que o MES saber

In [34]:
# ============================================================
# FUNÇÕES DO MÓDULO DE VIBRAÇÃO
# ============================================================

def sortear_leitura_vibracao(seed: int = None) -> dict:
    """
    Sorteia uma linha do CSV — simula uma leitura ao vivo do sensor ADXL345.
    Em produção: esse dicionário viria do sensor em tempo real.
    """
    linha = df_vibracao[feature_cols].sample(1, random_state=seed).iloc[0]
    return linha.to_dict()


def chamar_svm_vibracao(features: dict) -> dict:
    """
    Classifica uma janela de vibração com o SVM.

    Seleciona automaticamente as features corretas:
      - Se o scaler foi treinado com nomes de colunas (feature_names_in_),
        usa exatamente essas colunas na mesma ordem.
      - Caso contrário, usa as primeiras n_features_in_ features do dicionário.

    Retorna: { 'classe': 'yes' | 'no' }
    """
    n_esperado = scaler_vibracao.n_features_in_

    if hasattr(scaler_vibracao, 'feature_names_in_'):
        cols = list(scaler_vibracao.feature_names_in_)
        X = np.array([features[c] for c in cols]).reshape(1, -1)
    else:
        vals = list(features.values())[:n_esperado]
        X    = np.array(vals).reshape(1, -1)

    X_scaled = scaler_vibracao.transform(X)
    classe   = svm_vibracao.predict(X_scaled)[0]
    return {'classe': classe}


# ============================================================
# DEMONSTRAÇÃO — pipeline completo de vibração
# ============================================================

print('=== Pipeline de Vibração — passo a passo ===\n')
print(f'ℹ️  CSV tem {len(feature_cols)} features | Scaler espera {scaler_vibracao.n_features_in_} features')
if hasattr(scaler_vibracao, 'feature_names_in_'):
    print(f'   Features do scaler : {list(scaler_vibracao.feature_names_in_)}')
else:
    print(f'   Usando as primeiras {scaler_vibracao.n_features_in_} colunas do CSV')
print()

leitura   = sortear_leitura_vibracao(seed=42)
resultado = chamar_svm_vibracao(leitura)

print(f'PASSO 1 — Leitura sorteada do CSV (primeiras 4 features):')
print(f'  { {k: round(v,4) for k,v in list(leitura.items())[:4]} } ...')
print()
print(f'PASSO 2 — Normalização com scaler → vetor pronto para o SVM')
print()
print(f'PASSO 3 — SVM classificou como: "{resultado["classe"]}" '
      f'({"⚠️ defeito detectado" if resultado["classe"]=="yes" else "✅ operação normal"})')
print()
print('─' * 55)
print('💡 Na fábrica real: os números viriam do sensor ao vivo.')
print('   O resultado seria o mesmo: apenas "yes" ou "no".')
print('   Só isso interessa para o Motor de Inferência.')

=== Pipeline de Vibração — passo a passo ===

ℹ️  CSV tem 27 features | Scaler espera 7 features
   Usando as primeiras 7 colunas do CSV

PASSO 1 — Leitura sorteada do CSV (primeiras 4 features):
  {'X_rms': 2340205653356130.0, 'X_mean': -168125.0, 'X_std': 1.62787006161425e+16, 'X_max': 0.27} ...

PASSO 2 — Normalização com scaler → vetor pronto para o SVM

PASSO 3 — SVM classificou como: "no_defected" (✅ operação normal)

───────────────────────────────────────────────────────
💡 Na fábrica real: os números viriam do sensor ao vivo.
   O resultado seria o mesmo: apenas "yes" ou "no".
   Só isso interessa para o Motor de Inferência.


---
## 4. Módulo PCA — Detecção de Anomalia por Erro de Reconstrução

Terceiro módulo de IA integrado ao SBC, baseado na **Aula 8 (Redução de Dimensionalidade)**.

### Conceito

O PCA é treinado **apenas nos dados normais** (label = `no`). Quando uma leitura nova chega:
1. O scaler padroniza as features
2. O PCA projeta no espaço reduzido e reconstrói
3. Calcula o **erro de reconstrução (MSE)** entre o original e o reconstruído
4. Se MSE > limiar (P95 dos normais) → **anomalia detectada**

Dados "estranhos" (drift ou defeito) não são bem representados pelo PCA de referência, gerando erro alto.

```
features (27D) → scaler → PCA → reconstruir → MSE → anomalia?
                         ↑                          ↑
                  treinado só             limiar = P95 dos
                  em normais              erros normais
```

> 💡 Este módulo complementa o SVM: o SVM faz classificação binária (yes/no),
> enquanto o PCA detecta **drift** — padrões fora da distribuição normal, mesmo que o SVM não os classifique como erro.

In [35]:
# ============================================================
# MÓDULO PCA — Treinamento + Detecção de Anomalia
# Baseado na Aula 8 (Seção 8: Drift Temporal com PCA)
# ============================================================
# Como os artefatos .joblib da Aula 8 não foram exportados,
# treinamos o PCA aqui diretamente a partir do CSV de vibração
# que já está carregado (df_vibracao).
# ============================================================

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler as _StdScaler

# ── 1. Preparar dados ─────────────────────────────────────────
X_pca_all   = df_vibracao[feature_cols].values
y_pca_text  = df_vibracao['label'].values

# ── 2. Padronizar (scaler próprio, independente do SVM) ───────
scaler_pca = _StdScaler()
X_pca_scaled = scaler_pca.fit_transform(X_pca_all)

# ── 3. Determinar n_components ideal (≥ 95% variância) ───────
pca_full = PCA()
pca_full.fit(X_pca_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = int(np.argmax(cumvar >= 0.95) + 1)

# ── 4. Treinar PCA de referência APENAS em dados normais ─────
mask_normal = (y_pca_text == 'no')
X_normal  = X_pca_scaled[mask_normal]
X_anomalo = X_pca_scaled[~mask_normal]

# Limitar componentes para evitar reconstrução perfeita.
# Se n_comp == n_amostras, o PCA captura TODA a variância e
# o threshold fica 0 → tudo é "anomalia".
# Regra: usar no máximo (n_amostras - 2) componentes para que
# haja erro residual nos dados normais.
max_comp    = min(X_normal.shape[0], X_normal.shape[1])
n_comp_ref  = min(n_95, max(1, max_comp - 2))

print(f'ℹ️  n_95={n_95}, amostras_normais={X_normal.shape[0]}, '
      f'max_comp={max_comp} → usando n_components={n_comp_ref}')

pca_ref = PCA(n_components=n_comp_ref)
pca_ref.fit(X_normal)

# ── 5. Erro de reconstrução → limiar P95 ─────────────────────
X_normal_rec = pca_ref.inverse_transform(pca_ref.transform(X_normal))
mse_normal   = np.mean((X_normal - X_normal_rec) ** 2, axis=1)
threshold_pca = float(np.percentile(mse_normal, 95))

# ── 6. Validar com dados anômalos ────────────────────────────
X_anomalo_rec = pca_ref.inverse_transform(pca_ref.transform(X_anomalo))
mse_anomalo   = np.mean((X_anomalo - X_anomalo_rec) ** 2, axis=1)

# ── 7. Exportar artefatos (mesma estrutura da Aula 8) ────────
from pathlib import Path as _Path
EXPORT_PCA_DIR = _Path(r'modelos_exportados\reducao_dimensionalidade')
EXPORT_PCA_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(pca_ref,    EXPORT_PCA_DIR / 'pca_reference.joblib')
joblib.dump(scaler_pca, EXPORT_PCA_DIR / 'scaler_features.joblib')

config_pca = {
    'n_componentes_ref':      n_comp_ref,
    'n_componentes_95':       n_95,
    'threshold_anomalia_mse': threshold_pca,
    'feature_cols':           feature_cols,
    'n_amostras_normal':      int(X_normal.shape[0]),
    'n_amostras_anomalo':     int(X_anomalo.shape[0]),
    'n_features_original':    int(X_pca_scaled.shape[1]),
    'aula':                   'Projeto_Final (treinado inline)',
}
joblib.dump(config_pca, EXPORT_PCA_DIR / 'reducao_config.joblib')

# ── 8. Função de predição ─────────────────────────────────────

def chamar_pca_anomalia(features: dict) -> dict:
    """
    Avalia se uma leitura de vibração é anômala via erro de
    reconstrução PCA (treinado apenas em dados normais).

    Entrada : dicionário com as 27 features de uma janela
    Saída   : { 'anomalia': bool, 'mse': float, 'threshold': float }
    """
    n_esperado = scaler_pca.n_features_in_

    if hasattr(scaler_pca, 'feature_names_in_'):
        cols = list(scaler_pca.feature_names_in_)
        X = np.array([features[c] for c in cols]).reshape(1, -1)
    else:
        vals = list(features.values())[:n_esperado]
        X = np.array(vals).reshape(1, -1)

    X_scaled = scaler_pca.transform(X)
    X_rec    = pca_ref.inverse_transform(pca_ref.transform(X_scaled))
    mse      = float(np.mean((X_scaled - X_rec) ** 2))

    return {
        'anomalia':  mse > threshold_pca,
        'mse':       round(mse, 6),
        'threshold': round(threshold_pca, 6),
    }


# ============================================================
# DEMONSTRAÇÃO
# ============================================================

print('=' * 60)
print('📦 MÓDULO PCA — Detecção de Anomalia por Reconstrução')
print('=' * 60)
print(f'   Amostras normais   : {X_normal.shape[0]}')
print(f'   Amostras anômalas  : {X_anomalo.shape[0]}')
print(f'   Features originais : {X_pca_scaled.shape[1]}')
print(f'   Componentes (95%)  : {n_95}')
print(f'   Componentes (ref)  : {n_comp_ref}')
print(f'   Threshold (P95)    : {threshold_pca:.6f}')
print()
print(f'📊 Erro de Reconstrução:')
print(f'   Normal  → Média = {np.mean(mse_normal):.6f}, Std = {np.std(mse_normal):.6f}')
print(f'   Anomalia→ Média = {np.mean(mse_anomalo):.6f}, Std = {np.std(mse_anomalo):.6f}')
print(f'   Anomalias acima do limiar: {np.sum(mse_anomalo > threshold_pca)}/{len(mse_anomalo)}')
print()

# Exportação
print(f'📁 Artefatos exportados em: {EXPORT_PCA_DIR}')
for f in sorted(EXPORT_PCA_DIR.glob('*.joblib')):
    print(f'   ✅ {f.name} ({f.stat().st_size / 1024:.1f} KB)')
print()

# Demo com leitura sorteada
print('─' * 60)
print('DEMO — Leitura sorteada passada pelo PCA:')
leitura_demo = sortear_leitura_vibracao(seed=42)
res_pca = chamar_pca_anomalia(leitura_demo)
print(f'   MSE       : {res_pca["mse"]}')
print(f'   Threshold : {res_pca["threshold"]}')
print(f'   Anomalia  : {"🔴 DETECTADA" if res_pca["anomalia"] else "🟢 Normal"}')
print()
print('💡 O SVM classifica yes/no. O PCA detecta DRIFT — padrões fora')
print('   da distribuição normal, mesmo que o SVM não os classifique como erro.')

ℹ️  n_95=17, amostras_normais=7, max_comp=7 → usando n_components=5
📦 MÓDULO PCA — Detecção de Anomalia por Reconstrução
   Amostras normais   : 7
   Amostras anômalas  : 25
   Features originais : 27
   Componentes (95%)  : 17
   Componentes (ref)  : 5
   Threshold (P95)    : 0.305605

📊 Erro de Reconstrução:
   Normal  → Média = 0.088442, Std = 0.124563
   Anomalia→ Média = 0.938579, Std = 0.358556
   Anomalias acima do limiar: 25/25

📁 Artefatos exportados em: modelos_exportados\reducao_dimensionalidade
   ✅ pca_reference.joblib (2.3 KB)
   ✅ reducao_config.joblib (0.5 KB)
   ✅ scaler_features.joblib (1.2 KB)

────────────────────────────────────────────────────────────
DEMO — Leitura sorteada passada pelo PCA:
   MSE       : 1.007665
   Threshold : 0.305605
   Anomalia  : 🔴 DETECTADA

💡 O SVM classifica yes/no. O PCA detecta DRIFT — padrões fora
   da distribuição normal, mesmo que o SVM não os classifique como erro.


---
## 5. Grounding — Da IA ao Símbolo

Este é o **ponto de conexão** entre as duas disciplinas.

O motor não entende `0.91` ou `"yes"`. Ele entende `"confianca(alta)"` e `"vibracao(erro)"` — os mesmos símbolos das Aulas 3–5.

A função de grounding faz essa tradução, usando os limites definidos na Aula 3:

```
OUTPUT DA IA                  GROUNDING               SÍMBOLO (WM)
probability = 0.91   ───────────────────────────▶   confianca(alta)
tagName = leg_broken ───────────────────────────▶   evento_visao(leg_broken)
classe = "yes"       ───────────────────────────▶   vibracao(erro)
classe = "no"        ───────────────────────────▶   vibracao(normal)
anomalia = True      ───────────────────────────▶   anomalia_pca(detectada)
anomalia = False     ───────────────────────────▶   anomalia_pca(normal)
criticidade = alta   ───────────────────────────▶   criticidade(alta)
prazo = 10h          ───────────────────────────▶   prazo(curto)
```

> 💡 **Novo (Projeto Final):** O grounding agora inclui o módulo PCA de detecção de anomalia
> (Aula 8). O PCA complementa o SVM — detecta **drift** mesmo quando o SVM classifica como normal.

In [37]:
# ============================================================
# LIMITES DE GROUNDING — mesmos da Aula 3
# ============================================================

LIMITES = {
    'confianca_alta': 0.80,   # prob >= 0.80 → confianca(alta)
    'prazo_curto':    20,     # horas <= 20  → prazo(curto)
}

# Classes que o SVM pode retornar como "defeito detectado".
# Depende de como o modelo foi treinado — pode ser 'yes' ou 'defected'.
# O importante é que o grounding normalize para vibracao(erro) ou vibracao(normal).
_CLASSES_DEFEITO = {'yes', 'defected'}


def grounding_visao(output_azure: dict) -> List[str]:
    """
    Converte o output do Azure Custom Vision em símbolos para a WM.
    Entrada : { 'tag': 'leg_broken', 'probabilidade': 0.91 }
    Saída   : [ 'evento_visao(leg_broken)', 'confianca(alta)' ]
    """
    simbolos = [f"evento_visao({output_azure['tag']})"]
    if output_azure['probabilidade'] >= LIMITES['confianca_alta']:
        simbolos.append('confianca(alta)')
    else:
        simbolos.append('confianca(baixa)')
    return simbolos


def grounding_vibracao(output_svm: dict) -> List[str]:
    """
    Converte o output do SVM de vibração em símbolo para a WM.
    Aceita ambas as convenções de nomes de classe:
      - 'yes' / 'no'               (convenção do CSV)
      - 'defected' / 'no_defected' (convenção do modelo SVM)
    Saída   : [ 'vibracao(erro)' ] ou [ 'vibracao(normal)' ]
    """
    classe = output_svm['classe'].lower()
    return ['vibracao(erro)'] if classe in _CLASSES_DEFEITO else ['vibracao(normal)']


def grounding_pca(output_pca: dict) -> List[str]:
    """
    Converte o output do PCA de anomalia em símbolo para a WM.
    Entrada : { 'anomalia': True/False, 'mse': 0.0234, 'threshold': 0.0180 }
    Saída   : [ 'anomalia_pca(detectada)' ] ou [ 'anomalia_pca(normal)' ]

    💡 Complementa o SVM: detecta drift (padrões fora da distribuição normal)
       mesmo quando o SVM classifica como 'no'.
    """
    return ['anomalia_pca(detectada)'] if output_pca['anomalia'] else ['anomalia_pca(normal)']


def grounding_contexto(contexto: dict) -> List[str]:
    """
    Converte o contexto operacional em símbolos para a WM.
    Entrada : { 'criticidade': 'alta', 'prazo_horas': 10 }
    Saída   : [ 'criticidade(alta)', 'prazo(curto)' ]
    """
    simbolos = [f"criticidade({contexto['criticidade']})"]
    simbolos.append('prazo(curto)' if contexto['prazo_horas'] <= LIMITES['prazo_curto'] else 'prazo(longo)')
    return simbolos


def construir_fatos(output_azure: dict, output_svm: dict,
                    contexto: dict, output_pca: dict = None) -> List[str]:
    """
    Une os quatro módulos de grounding em uma lista de fatos simbólicos.
    Essa lista é a entrada direta da WorkingMemory do Motor de Inferência.

    output_pca é opcional para manter retrocompatibilidade com testes antigos.
    Se fornecido, inclui anomalia_pca(detectada) ou anomalia_pca(normal).
    """
    fatos = grounding_visao(output_azure) + grounding_vibracao(output_svm) + grounding_contexto(contexto)
    if output_pca is not None:
        fatos += grounding_pca(output_pca)
    return fatos


# --- Demonstração passo a passo ---
print('=== Grounding passo a passo (com PCA) ===')
print(f'   Classes SVM reconhecidas como defeito: {_CLASSES_DEFEITO}')
print(f'   Classes reais do SVM: {list(svm_vibracao.classes_)}')
print()

out_azure_ex = {'tag': 'leg_broken', 'probabilidade': 0.91}
out_svm_ex   = {'classe': 'no'}
out_pca_ex   = {'anomalia': True, 'mse': 0.0234, 'threshold': 0.0180}
ctx_ex       = {'criticidade': 'alta', 'prazo_horas': 10}

print(f"  Azure    : tag={out_azure_ex['tag']}, prob={out_azure_ex['probabilidade']}")
print(f"  SVM      : classe={out_svm_ex['classe']}")
print(f"  PCA      : anomalia={out_pca_ex['anomalia']}, mse={out_pca_ex['mse']}")
print(f"  Contexto : {ctx_ex}")
print()
print(f"  → visão     : {grounding_visao(out_azure_ex)}")
print(f"  → vibração  : {grounding_vibracao(out_svm_ex)}")
print(f"  → PCA       : {grounding_pca(out_pca_ex)}")
print(f"  → contexto  : {grounding_contexto(ctx_ex)}")
print()
print(f"  ✅ WM inicial: {construir_fatos(out_azure_ex, out_svm_ex, ctx_ex, out_pca_ex)}")

=== Grounding passo a passo (com PCA) ===
   Classes SVM reconhecidas como defeito: {'yes', 'defected'}
   Classes reais do SVM: [np.str_('defected'), np.str_('no_defected')]

  Azure    : tag=leg_broken, prob=0.91
  SVM      : classe=no
  PCA      : anomalia=True, mse=0.0234
  Contexto : {'criticidade': 'alta', 'prazo_horas': 10}

  → visão     : ['evento_visao(leg_broken)', 'confianca(alta)']
  → vibração  : ['vibracao(normal)']
  → PCA       : ['anomalia_pca(detectada)']
  → contexto  : ['criticidade(alta)', 'prazo(curto)']

  ✅ WM inicial: ['evento_visao(leg_broken)', 'confianca(alta)', 'vibracao(normal)', 'criticidade(alta)', 'prazo(curto)', 'anomalia_pca(detectada)']


---
## 6. Gerando os Fatos Reais — Chamando os Três Modelos + Contexto

Agora conectamos tudo.

Para cada amostra vamos:
1. **Enviar a imagem** para o Azure → receber a classificação visual real
2. **Sortear uma leitura** do CSV de vibração → passar para o SVM → receber `yes/no`
3. **Mesma leitura** → passar para o PCA → receber `anomalia/normal` (Aula 8)
4. **Aplicar o grounding** nos três resultados + contexto operacional
5. **Montar a lista de fatos** que entra na WorkingMemory do motor

```
imagem.jpg    ──▶ Azure API ──▶ { tag, prob }
                                       │
linha do CSV  ──▶ SVM       ──▶ { classe }          │
              ──▶ PCA       ──▶ { anomalia, mse }   │
                                       │             │
contexto MES  ─────────────────────────│─────────────│
                                       ▼             │
                               grounding()           │
                                       │             │
                                       ▼
     [ evento_visao(...), confianca(...), vibracao(...),
       anomalia_pca(...), criticidade(...), prazo(...) ]
```

> 💡 **Novo no Projeto Final:** A leitura de vibração alimenta **dois módulos** —
> o SVM (classificação) e o PCA (detecção de drift). Juntos, cobrem falha direta e degradação gradual.

In [20]:
# ============================================================
# UPLOAD D — Imagens das peças para o Azure
# (Adaptado para VS Code — usa caminhos locais do archive/)
# ============================================================
# Mapeamento: ID da peça → imagem correspondente
# O ID = nome do ficheiro (sem extensão), igual ao CSV contexto_pecas
# Cobertura: todas as 5 classes do Azure + variação de criticidade/prazo
#
# | ID                | Classe esperada  | Criticidade | Prazo  |
# |-------------------|-----------------|-------------|--------|
# | scratch_4_3       | no_defected     | alta        | 45h    |
# | scratch_4_4       | no_defected     | media       | 20h    |
# | scratch_3_4       | no_defected     | baixa       | 40h    |
# | scratch_3_5       | no_defected     | media       | 30h    |
# | bed_not_stick_1   | bed_no_stick    | alta        | 52h    |
# | bed_not_stick_0   | bed_no_stick    | alta        | 38h    |
# | bed_not_stick_10  | bed_no_stick    | baixa       | 24h    |
# | no_support_34     | no_support      | media       | 63h    |
# | no_support_234    | no_support      | baixa       | 22h    |
# | no_support_235    | no_support      | media       | 15h    |
# | no_support_236    | no_support      | alta        | 27h    |
# | leg_broken_118    | leg_broken      | baixa       | 68h    |
# | leg_broken_119    | leg_broken      | alta        | 55h    |
# | leg_broken_120    | leg_broken      | media       | 29h    |
# | no_bottom_4       | no_bottom       | baixa       | 33h    |
# | no_bottom_6       | no_bottom       | baixa       | 24h    |
# | no_bottom_5       | no_bottom       | media       | 18h    |

import os
from pathlib import Path

ARCHIVE_PATH = Path(r'archive')

IMAGENS_SELECIONADAS = [
    # --- no_defected (4 imagens) — classe "scratch" = sem defeito ---
    ('scratch_4_3',      ARCHIVE_PATH / 'no_defected' / 'scratch_4_3.jpg'),
    ('scratch_4_4',      ARCHIVE_PATH / 'no_defected' / 'scratch_4_4.jpg'),
    ('scratch_3_4',      ARCHIVE_PATH / 'no_defected' / 'scratch_3_4.jpg'),
    ('scratch_3_5',      ARCHIVE_PATH / 'no_defected' / 'scratch_3_5.jpg'),

    # --- bed_no_stick (3 imagens) — falha de adesão ---
    ('bed_not_stick_1',  ARCHIVE_PATH / 'defected' / 'bed_not_stick_1.jpg'),
    ('bed_not_stick_0',  ARCHIVE_PATH / 'defected' / 'bed_not_stick_0.jpg'),
    ('bed_not_stick_10', ARCHIVE_PATH / 'defected' / 'bed_not_stick_10.jpg'),

    # --- no_support (4 imagens) — suporte ausente ---
    ('no_support_34',    ARCHIVE_PATH / 'defected' / 'no_support_34.jpg'),
    ('no_support_234',   ARCHIVE_PATH / 'defected' / 'no_support_234.jpg'),
    ('no_support_235',   ARCHIVE_PATH / 'defected' / 'no_support_235.jpg'),
    ('no_support_236',   ARCHIVE_PATH / 'defected' / 'no_support_236.jpg'),

    # --- leg_broken (3 imagens) — fratura estrutural ---
    ('leg_broken_118',   ARCHIVE_PATH / 'defected' / 'leg_broken_118.jpg'),
    ('leg_broken_119',   ARCHIVE_PATH / 'defected' / 'leg_broken_119.jpg'),
    ('leg_broken_120',   ARCHIVE_PATH / 'defected' / 'leg_broken_120.jpg'),

    # --- no_bottom (3 imagens) — fundo ausente ---
    ('no_bottom_4',      ARCHIVE_PATH / 'defected' / 'no_bottom_4.jpg'),
    ('no_bottom_6',      ARCHIVE_PATH / 'defected' / 'no_bottom_6.jpg'),
    ('no_bottom_5',      ARCHIVE_PATH / 'defected' / 'no_bottom_5.jpg'),
]

IMAGENS = []
print('📂 Carregando imagens locais...\n')

for pid, caminho in IMAGENS_SELECIONADAS:
    caminho = Path(caminho)
    if caminho.exists():
        IMAGENS.append({'id': pid, 'imagem': str(caminho)})
        print(f'   ✅ {pid}: {caminho.name}')
    else:
        print(f'   ⚠️  {pid}: arquivo não encontrado — {caminho}')

IMAGENS = sorted(IMAGENS, key=lambda x: x['id'])

print(f'\n{len(IMAGENS)} imagem(ns) prontas para enviar ao Azure.')
print(f'   IDs carregados : {[img["id"] for img in IMAGENS]}')
print(f'   IDs no contexto: {list(df_contexto.index)}')

# Avisa se algum ID não bate com o contexto
ids_sem_contexto = [img['id'] for img in IMAGENS if img['id'] not in df_contexto.index]
if ids_sem_contexto:
    print(f'\n⚠️  IDs sem contexto no CSV: {ids_sem_contexto}')
    print('   Adicione esses IDs ao contexto_pecas.csv ou ajuste os IDs acima.')
else:
    print('\n✅ Todos os IDs encontrados no contexto — pronto para rodar!')

📂 Carregando imagens locais...

   ✅ scratch_4_3: scratch_4_3.jpg
   ✅ scratch_4_4: scratch_4_4.jpg
   ✅ scratch_3_4: scratch_3_4.jpg
   ✅ scratch_3_5: scratch_3_5.jpg
   ✅ bed_not_stick_1: bed_not_stick_1.jpg
   ✅ bed_not_stick_0: bed_not_stick_0.jpg
   ✅ bed_not_stick_10: bed_not_stick_10.jpg
   ✅ no_support_34: no_support_34.jpg
   ✅ no_support_234: no_support_234.jpg
   ✅ no_support_235: no_support_235.jpg
   ✅ no_support_236: no_support_236.jpg
   ✅ leg_broken_118: leg_broken_118.jpg
   ✅ leg_broken_119: leg_broken_119.jpg
   ✅ leg_broken_120: leg_broken_120.jpg
   ✅ no_bottom_4: no_bottom_4.jpg
   ✅ no_bottom_6: no_bottom_6.jpg
   ✅ no_bottom_5: no_bottom_5.jpg

17 imagem(ns) prontas para enviar ao Azure.
   IDs carregados : ['bed_not_stick_0', 'bed_not_stick_1', 'bed_not_stick_10', 'leg_broken_118', 'leg_broken_119', 'leg_broken_120', 'no_bottom_4', 'no_bottom_5', 'no_bottom_6', 'no_support_234', 'no_support_235', 'no_support_236', 'no_support_34', 'scratch_3_4', 'scratch_3_5', 

In [40]:
# ============================================================
# GERANDO OS FATOS REAIS — Azure + SVM + PCA + Contexto
# ============================================================

FATOS_REAIS    = {}
OUTPUTS_BRUTOS = {}

print('Chamando Azure + SVM + PCA e construindo fatos...\n')
print(f'{"ID":<20} {"Azure (tag)":<16} {"Prob":>5}  {"Conf":<6}  {"SVM":>6}  {"PCA":<10} {"Crit":<8} {"Prazo"}')
print('-' * 105)

for i, item in enumerate(IMAGENS):
    pid = item['id']

    # Verifica se o ID da imagem existe no contexto (MES)
    if pid not in df_contexto.index:
        print(f'⚠️  ID "{pid}" não encontrado no contexto_pecas.csv — pulando.')
        continue

    # 1. Visão: chamada real à API do Azure
    out_azure = chamar_azure(item['imagem'])

    # 2. Vibração: sorteia linha do CSV → SVM
    #    Em produção: viria do sensor ADXL345 ao vivo
    leitura_vib = sortear_leitura_vibracao(seed=i)
    out_svm     = chamar_svm_vibracao(leitura_vib)

    # 3. PCA: mesma leitura → detecção de anomalia por reconstrução
    out_pca = chamar_pca_anomalia(leitura_vib)

    # 4. Contexto: vem do CSV que simula o MES
    ctx = df_contexto.loc[pid].to_dict()

    # 5. Grounding → fatos simbólicos (agora com 4 módulos)
    fatos = construir_fatos(out_azure, out_svm, ctx, out_pca)

    FATOS_REAIS[pid]    = fatos
    OUTPUTS_BRUTOS[pid] = {'azure': out_azure, 'svm': out_svm, 'pca': out_pca, 'contexto': ctx}

    conf = 'alta' if out_azure['probabilidade'] >= LIMITES['confianca_alta'] else 'baixa'
    pca_str = '🔴 drift' if out_pca['anomalia'] else '🟢 ok'
    print(f"{pid:<20} {out_azure['tag']:<16} {out_azure['probabilidade']:>5.2f}  {conf:<6}  "
          f"{out_svm['classe']:>6}  {pca_str:<10} {ctx['criticidade']:<8} {ctx['prazo_horas']}h")

print('\n✅ Fatos reais construídos!')
print()
print('Origem de cada dado:')
print('  Azure    → tag + probabilidade   (API real)')
print('  SVM      → yes / no              (modelo real + linha do CSV)')
print('  PCA      → anomalia / normal     (erro de reconstrução — Aula 8)')
print('  Contexto → criticidade + prazo   (CSV que simula o MES)')

Chamando Azure + SVM + PCA e construindo fatos...

ID                   Azure (tag)       Prob  Conf       SVM  PCA        Crit     Prazo
---------------------------------------------------------------------------------------------------------
bed_not_stick_0      bed_no_stick      0.88  alta    no_defected  🔴 drift    alta     38h
bed_not_stick_1      bed_no_stick      0.44  baixa   no_defected  🔴 drift    alta     52h
bed_not_stick_10     bed_no_stick      0.37  baixa   no_defected  🔴 drift    baixa    24h
leg_broken_118       no_bottom         0.61  baixa   no_defected  🔴 drift    baixa    68h
leg_broken_119       no_bottom         0.51  baixa   no_defected  🔴 drift    alta     55h
leg_broken_120       leg_broken        0.35  baixa   no_defected  🔴 drift    media    29h
no_bottom_4          no_bottom         0.56  baixa   no_defected  🔴 drift    baixa    33h
no_bottom_5          no_bottom         0.44  baixa   no_defected  🔴 drift    media    18h
no_bottom_6          no_bottom      

---
## 7. Base de Conhecimento — 21 Regras (Cobertura Completa + PCA)

Todas as 5 classes do Azure Custom Vision + módulo PCA de anomalia estão cobertas.

---

### Camada 1 — Interpretação dos sensores (genérica)
```
R01: SE vibracao(erro)             → risco_mecanico(alto)
R02: SE confianca(alta)            → sinal_visual_confiavel(sim)
R03: SE vibracao(normal)           → risco_mecanico(baixo)
R19: SE anomalia_pca(detectada)    → drift_detectado(sim)        ← NOVO (Aula 8)
```

### Camada 2 — Decisões base
```
R04: SE risco_mecanico(alto) E criticidade(alta)            → decisao(NOGO)
R05: SE risco_mecanico(alto) E criticidade(media)           → decisao(INVESTIGAR)
R06: SE evento_visao(no_defected) E vibracao(normal)        → decisao(GO)
```

### Classe A — leg_broken (fratura estrutural)
```
R07: SE evento_visao(leg_broken) E sinal_visual_confiavel(sim) → suporte_quebrado(confirmado)
R08: SE suporte_quebrado(confirmado) E criticidade(alta)       → decisao(NOGO)
```

### Conflito de sensores
```
R09: SE evento_visao(leg_broken) E confianca(baixa) E vibracao(erro) → decisao(INVESTIGAR)
```

### Classe B — bed_no_stick (falha de adesão)
```
R10: SE evento_visao(bed_no_stick) E sinal_visual_confiavel(sim) → sem_fixacao(confirmado)
R11: SE sem_fixacao(confirmado) E criticidade(alta)              → decisao(NOGO)
R12: SE sem_fixacao(confirmado) E criticidade(media) E prazo(curto) → decisao(AJUSTAR)
```

### Classe C — no_support (suporte ausente)
```
R13: SE evento_visao(no_support) E sinal_visual_confiavel(sim)     → suporte_ausente(confirmado)
R14: SE suporte_ausente(confirmado) E criticidade(alta)            → decisao(NOGO)
R15: SE suporte_ausente(confirmado) E criticidade(media) E prazo(longo) → decisao(INVESTIGAR)
```

### Classe D — no_bottom (fundo ausente)
```
R16: SE evento_visao(no_bottom) E sinal_visual_confiavel(sim)      → base_inexistente(confirmado)
R17: SE base_inexistente(confirmado) E criticidade(alta)           → decisao(NOGO)
R18: SE base_inexistente(confirmado) E criticidade(baixa) E prazo(longo) → decisao(MONITORAR)
```

### PCA — Detecção de Drift (Aula 8) ← NOVO
```
R19: SE anomalia_pca(detectada)                                     → drift_detectado(sim)
R20: SE drift_detectado(sim) E vibracao(normal) E criticidade(alta) → decisao(INVESTIGAR)
R21: SE drift_detectado(sim) E vibracao(normal) E criticidade(baixa)→ decisao(MONITORAR)
```

> 💡 **R20 e R21 capturam o cenário mais interessante:** o PCA detectou drift (degradação gradual)
> mas o SVM ainda classifica como normal. É um **alerta precoce** — o sistema está saindo da
> distribuição normal antes de falhar de fato. Peça crítica → investigar; peça simples → monitorar.

---

### Ações cobertas

| Ação | Regras que a produzem |
|---|---|
| **GO** | R06 |
| **MONITORAR** | R18, **R21** |
| **AJUSTAR** | R12 |
| **INVESTIGAR** | R05, R09, R15, **R20** |
| **NOGO** | R04, R08, R11, R14, R17 |

In [41]:
# ============================================================
# BASE DE CONHECIMENTO — Projeto Final (expandida com PCA)
# Regras criadas para os dados reais do Azure + SVM + PCA
# Cobertura: TODAS as 5 classes do Azure + detecção de drift
# ============================================================

# a acao é o que fazer
# e a prioridade é quanto confiar nessa regra em relação às outras
# ambas são conhecimento humano formalizado — não saem dos dados, saem de quem entende do processo.

BASE_REGRAS: List[Rule] = [

    # ═══════════════════════════════════════════════════════════
    # CAMADA 1: Interpretação dos sensores (genéricas)
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R01_VIBRACAO_ERRO_RISCO_ALTO',
        condicoes=['vibracao(erro)'],
        acao='risco_mecanico(alto)',
        prioridade=95,
        justificativa='Vibração em erro indica anomalia mecânica grave.'
    ),
    Rule(
        nome='R02_SINAL_VISUAL_CONFIAVEL',
        condicoes=['confianca(alta)'],
        acao='sinal_visual_confiavel(sim)',
        prioridade=85,
        justificativa='Confiança alta: predição do Azure é confiável.'
    ),
    Rule(
        nome='R03_VIBRACAO_NORMAL_RISCO_BAIXO',
        condicoes=['vibracao(normal)'],
        acao='risco_mecanico(baixo)',
        prioridade=60,
        justificativa='Vibração normal — sem anomalia mecânica detectada.'
    ),

    # ═══════════════════════════════════════════════════════════
    # CAMADA 2: Decisões base (risco mecânico + criticidade)
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R04_NOGO_RISCO_ALTO_CRITICO',
        condicoes=['risco_mecanico(alto)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=99,
        justificativa='Risco mecânico alto em peça crítica: parar imediatamente.'
    ),
    Rule(
        nome='R05_INVESTIGAR_RISCO_ALTO_MEDIA',
        condicoes=['risco_mecanico(alto)', 'criticidade(media)'],
        acao='decisao(INVESTIGAR)',
        prioridade=88,
        justificativa='Risco alto em peça de criticidade média: investigar antes de decidir.'
    ),
    Rule(
        nome='R06_GO_TUDO_NORMAL',
        condicoes=['evento_visao(no_defected)', 'vibracao(normal)'],
        acao='decisao(GO)',
        prioridade=70,
        justificativa='Azure diz sem defeito e vibração normal: peça aprovada.'
    ),

    # ═══════════════════════════════════════════════════════════
    # CLASSE A: leg_broken (fratura estrutural)
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R07_LEG_BROKEN_CONFIRMADO',
        condicoes=['evento_visao(leg_broken)', 'sinal_visual_confiavel(sim)'],
        acao='suporte_quebrado(confirmado)',
        prioridade=92,
        justificativa='Azure detectou fratura com alta confiança: defeito estrutural confirmado.'
    ),
    Rule(
        nome='R08_NOGO_DEFEITO_ESTRUTURAL_CRITICO',
        condicoes=['suporte_quebrado(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=98,
        justificativa='Defeito estrutural confirmado em peça crítica: NOGO imediato.'
    ),
    Rule(
        nome='R22_INVESTIGAR_DEFEITO_ESTRUTURAL_MEDIO',
        condicoes=['suporte_quebrado(confirmado)', 'criticidade(media)'],
        acao='decisao(INVESTIGAR)',
        prioridade=86,
        justificativa='Fratura confirmada em peça de criticidade média: investigar causa e impacto antes de liberar.'
    ),
    Rule(
        nome='R23_MONITORAR_DEFEITO_ESTRUTURAL_BAIXO',
        condicoes=['suporte_quebrado(confirmado)', 'criticidade(baixa)'],
        acao='decisao(MONITORAR)',
        prioridade=73,
        justificativa='Fratura confirmada, mas peça não crítica: monitorar evolução sem parar produção.'
    ),

    # ═══════════════════════════════════════════════════════════
    # CONFLITO: visão incerta + vibração com erro
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R09_CONFLITO_VISAO_INCERTA_VIBRACAO_ERRO',
        condicoes=['evento_visao(leg_broken)', 'confianca(baixa)', 'vibracao(erro)'],
        acao='decisao(INVESTIGAR)',
        prioridade=89,
        justificativa='Azure viu fratura com baixa confiança, mas vibração confirma anomalia: investigar.'
    ),

    # ═══════════════════════════════════════════════════════════
    # CLASSE B: bed_no_stick (falha de adesão à base)
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R10_BED_NO_STICK_CONFIRMADO',
        condicoes=['evento_visao(bed_no_stick)', 'sinal_visual_confiavel(sim)'],
        acao='sem_fixacao(confirmado)',
        prioridade=90,
        justificativa='Azure detectou falha de adesão com confiança alta.'
    ),
    Rule(
        nome='R11_NOGO_ADESAO_CONFIRMADA_CRITICIDADE_ALTA',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=97,
        justificativa='Falha de adesão em peça crítica: interromper produção.'
    ),
    Rule(
        nome='R12_AJUSTAR_ADESAO_MEDIA_PRAZO_CURTO',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(media)', 'prazo(curto)'],
        acao='decisao(AJUSTAR)',
        prioridade=86,
        justificativa='Criticidade média e prazo curto: ajustar processo rapidamente.'
    ),
    Rule(
        nome='R24_INVESTIGAR_ADESAO_MEDIA_PRAZO_LONGO',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(media)', 'prazo(longo)'],
        acao='decisao(INVESTIGAR)',
        prioridade=84,
        justificativa='Falha de adesão em peça média com prazo longo: há tempo para investigar causa raiz.'
    ),
    Rule(
        nome='R25_MONITORAR_ADESAO_BAIXA',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(baixa)'],
        acao='decisao(MONITORAR)',
        prioridade=72,
        justificativa='Falha de adesão em peça não crítica: monitorar sem parar produção.'
    ),

    # ═══════════════════════════════════════════════════════════
    # CLASSE C: no_support (ausência de suporte estrutural)
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R13_NO_SUPPORT_CONFIRMADO',
        condicoes=['evento_visao(no_support)', 'sinal_visual_confiavel(sim)'],
        acao='suporte_ausente(confirmado)',
        prioridade=90,
        justificativa='Ausência de suporte estrutural confirmada com alta confiança.'
    ),
    Rule(
        nome='R14_NOGO_NO_SUPPORT_CRITICIDADE_ALTA',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=98,
        justificativa='Sem suporte em item crítico: bloqueio imediato (NOGO).'
    ),
    Rule(
        nome='R15_INVESTIGAR_SEM_SUPORTE_MEDIA_PRAZO_LONGO',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(media)', 'prazo(longo)'],
        acao='decisao(INVESTIGAR)',
        prioridade=88,
        justificativa='Com criticidade média e prazo longo, investigar causa raiz.'
    ),
    Rule(
        nome='R26_AJUSTAR_SEM_SUPORTE_MEDIA_PRAZO_CURTO',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(media)', 'prazo(curto)'],
        acao='decisao(AJUSTAR)',
        prioridade=85,
        justificativa='Suporte ausente em peça média com prazo curto: ajustar imediatamente para não perder entrega.'
    ),
    Rule(
        nome='R27_MONITORAR_SEM_SUPORTE_BAIXA',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(baixa)'],
        acao='decisao(MONITORAR)',
        prioridade=74,
        justificativa='Suporte ausente em peça não crítica: monitorar e reinspecionar na próxima janela.'
    ),

    # ═══════════════════════════════════════════════════════════
    # CLASSE D: no_bottom (fundo da peça ausente)
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R16_NO_BOTTOM_CONFIRMADO',
        condicoes=['evento_visao(no_bottom)', 'sinal_visual_confiavel(sim)'],
        acao='base_inexistente(confirmado)',
        prioridade=90,
        justificativa='Fundo ausente detectado com confiança alta.'
    ),
    Rule(
        nome='R17_NOGO_NO_BOTTOM_CRITICIDADE_ALTA',
        condicoes=['base_inexistente(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=96,
        justificativa='Ausência de fundo em peça crítica: NOGO.'
    ),
    Rule(
        nome='R28_AJUSTAR_NO_BOTTOM_MEDIA_PRAZO_CURTO',
        condicoes=['base_inexistente(confirmado)', 'criticidade(media)', 'prazo(curto)'],
        acao='decisao(AJUSTAR)',
        prioridade=84,
        justificativa='Fundo ausente em peça média com prazo curto: corrigir parâmetros antes da próxima camada.'
    ),
    Rule(
        nome='R29_INVESTIGAR_NO_BOTTOM_MEDIA_PRAZO_LONGO',
        condicoes=['base_inexistente(confirmado)', 'criticidade(media)', 'prazo(longo)'],
        acao='decisao(INVESTIGAR)',
        prioridade=83,
        justificativa='Fundo ausente em peça média com prazo longo: investigar causa antes de intervir.'
    ),
    Rule(
        nome='R18_MONITORAR_NO_BOTTOM_BAIXA_PRAZO_LONGO',
        condicoes=['base_inexistente(confirmado)', 'criticidade(baixa)', 'prazo(longo)'],
        acao='decisao(MONITORAR)',
        prioridade=82,
        justificativa='Baixa criticidade e prazo longo: monitorar e reinspecionar.'
    ),
    Rule(
        nome='R30_MONITORAR_NO_BOTTOM_BAIXA_PRAZO_CURTO',
        condicoes=['base_inexistente(confirmado)', 'criticidade(baixa)', 'prazo(curto)'],
        acao='decisao(MONITORAR)',
        prioridade=71,
        justificativa='Fundo ausente em peça não crítica: monitorar mesmo com prazo curto — impacto limitado.'
    ),

    # ═══════════════════════════════════════════════════════════
    # PCA: Detecção de Drift — Aula 8 (Redução de Dimensionalidade)
    # O PCA complementa o SVM: detecta degradação gradual
    # (drift) mesmo antes de o SVM classificar como erro.
    # ═══════════════════════════════════════════════════════════
    Rule(
        nome='R19_PCA_DRIFT_DETECTADO',
        condicoes=['anomalia_pca(detectada)'],
        acao='drift_detectado(sim)',
        prioridade=91,
        justificativa='PCA detectou erro de reconstrução acima do limiar: padrão fora da distribuição normal.'
    ),
    Rule(
        nome='R20_INVESTIGAR_DRIFT_SVM_NORMAL_CRITICO',
        condicoes=['drift_detectado(sim)', 'vibracao(normal)', 'criticidade(alta)'],
        acao='decisao(INVESTIGAR)',
        prioridade=87,
        justificativa='PCA detectou drift mas SVM diz normal — alerta precoce em peça crítica: investigar.'
    ),
    Rule(
        nome='R31_MONITORAR_DRIFT_SVM_NORMAL_MEDIO',
        condicoes=['drift_detectado(sim)', 'vibracao(normal)', 'criticidade(media)'],
        acao='decisao(MONITORAR)',
        prioridade=76,
        justificativa='PCA detectou drift mas SVM diz normal — peça de criticidade média: monitorar evolução.'
    ),
    Rule(
        nome='R21_MONITORAR_DRIFT_SVM_NORMAL_BAIXA',
        condicoes=['drift_detectado(sim)', 'vibracao(normal)', 'criticidade(baixa)'],
        acao='decisao(MONITORAR)',
        prioridade=75,
        justificativa='PCA detectou drift mas SVM diz normal — peça simples: acompanhar evolução.'
    ),
]

print(f'✅ Base de Conhecimento: {len(BASE_REGRAS)} regras')
print(f'   Camada 1 (interpretação)    : R01–R03, R19 (4 regras — inclui PCA)')
print(f'   Camada 2 (decisões base)    : R04–R06 (3 regras)')
print(f'   leg_broken                  : R07–R08, R22–R23 (NOGO + INVESTIGAR + MONITORAR)')
print(f'   conflito sensores           : R09     (visão incerta + vibração erro)')
print(f'   bed_no_stick                : R10–R12, R24–R25 (NOGO + AJUSTAR + INVESTIGAR + MONITORAR)')
print(f'   no_support                  : R13–R15, R26–R27 (NOGO + INVESTIGAR + AJUSTAR + MONITORAR)')
print(f'   no_bottom                   : R16–R18, R28–R30 (NOGO + AJUSTAR + INVESTIGAR + MONITORAR)')
print(f'   PCA drift (Aula 8)          : R19–R21, R31 (interpretação + INVESTIGAR + MONITORAR × 3)')
print(f'\n   Classes cobertas: leg_broken, bed_no_stick, no_support, no_bottom, no_defected')
print(f'   Cobertura: todas as combinações de criticidade/prazo do contexto_pecas.csv')

✅ Base de Conhecimento: 21 regras
   Camada 1 (interpretação)    : R01–R03, R19 (4 regras — inclui PCA)
   Camada 2 (decisões base)    : R04–R06 (3 regras)
   leg_broken                  : R07–R08 (confirmação + NOGO)
   conflito sensores           : R09     (visão incerta + vibração erro)
   bed_no_stick                : R10–R12 (confirmação + NOGO + AJUSTAR)
   no_support                  : R13–R15 (confirmação + NOGO + INVESTIGAR)
   no_bottom                   : R16–R18 (confirmação + NOGO + MONITORAR)
   PCA drift (Aula 8)          : R19–R21 (interpretação + INVESTIGAR + MONITORAR)

   Classes cobertas: leg_broken, bed_no_stick, no_support, no_bottom, no_defected
   Módulo PCA: drift detectado → alerta precoce (complementa SVM)


---
## 8. Rodando o Motor com Fatos Reais

Chegamos ao momento central: **o motor recebe fatos que vieram de IAs reais**.

Ele não sabe — e não precisa saber — se os fatos vieram de uma API do Azure ou de um `.joblib`. Ele apenas aplica as regras.

> 💡 Esta é a separação entre **percepção** (modelos de ML, sensores) e **raciocínio** (motor, BK).  
> Sistemas especialistas industriais sérios são construídos exatamente assim.

In [42]:
# ============================================================
# HELPERS — mesmos padrões das Aulas 4/5
# ============================================================

def rodar_motor(pid: str, verbose: bool = True) -> Tuple[InferenceEngine, Dict]:
    fatos = FATOS_REAIS[pid]
    wm    = WorkingMemory(fatos)
    eng   = InferenceEngine(BASE_REGRAS)
    if verbose:
        print(f"\n{'='*60}")
        print(f"  CASO {pid} | Fatos iniciais: {sorted(fatos)}")
        print(f"{'='*60}")
    resultado = eng.executar(wm, verbose=verbose)
    return eng, resultado


def resumo(pid: str, resultado: Dict):
    out = OUTPUTS_BRUTOS[pid]
    pca_info = out.get('pca', {})
    pca_str  = f"anomalia={pca_info.get('anomalia','N/A')}, mse={pca_info.get('mse','N/A')}"
    print(f'\n--- RESUMO {pid} ---')
    print(f"  Azure   : {out['azure']['tag']} (prob={out['azure']['probabilidade']:.2f})")
    print(f"  SVM     : {out['svm']['classe']}")
    print(f"  PCA     : {pca_str}")
    print(f"  Decisão : {resultado['decisao']}")
    print(f"  WM final: {sorted(resultado['wm_final'])}")


# ============================================================
# Mostra os IDs disponíveis para teste
# ============================================================
print('IDs disponíveis em FATOS_REAIS:')
print(f'  {list(FATOS_REAIS.keys())}')
print()

# ============================================================
# Primeiro caso disponível: teste completo com por_que()
# ============================================================
if FATOS_REAIS:
    pid_teste = list(FATOS_REAIS.keys())[0]
    eng_t, res_t = rodar_motor(pid_teste, verbose=True)
    resumo(pid_teste, res_t)
    print('\nÁrvore por_que():')
    print(json.dumps(eng_t.por_que(res_t['decisao']), indent=2, ensure_ascii=False))
else:
    print('⚠️ Nenhum fato real disponível.')

IDs disponíveis em FATOS_REAIS:
  ['bed_not_stick_0', 'bed_not_stick_1', 'bed_not_stick_10', 'leg_broken_118', 'leg_broken_119', 'leg_broken_120', 'no_bottom_4', 'no_bottom_5', 'no_bottom_6', 'no_support_234', 'no_support_235', 'no_support_236', 'no_support_34', 'scratch_3_4', 'scratch_3_5', 'scratch_4_3', 'scratch_4_4']


  CASO bed_not_stick_0 | Fatos iniciais: ['anomalia_pca(detectada)', 'confianca(alta)', 'criticidade(alta)', 'evento_visao(bed_no_stick)', 'prazo(longo)', 'vibracao(normal)']
  Ciclo 1: FIRE 'R19_PCA_DRIFT_DETECTADO' → 'drift_detectado(sim)'
  Ciclo 2: FIRE 'R20_INVESTIGAR_DRIFT_SVM_NORMAL_CRITICO' → 'decisao(INVESTIGAR)'
  Ciclo 3: FIRE 'R02_SINAL_VISUAL_CONFIAVEL' → 'sinal_visual_confiavel(sim)'
  Ciclo 4: FIRE 'R10_BED_NO_STICK_CONFIRMADO' → 'sem_fixacao(confirmado)'
  Ciclo 5: FIRE 'R11_NOGO_ADESAO_CONFIRMADA_CRITICIDADE_ALTA' → 'decisao(NOGO)'
  Ciclo 6: FIRE 'R03_VIBRACAO_NORMAL_RISCO_BAIXO' → 'risco_mecanico(baixo)'
  Ciclo 7: agenda vazia — motor para.

--- R

In [43]:
# ============================================================
# Segundo caso: teste contrastante (se disponível)
# ============================================================
if len(FATOS_REAIS) >= 2:
    pid_teste2 = list(FATOS_REAIS.keys())[1]
    eng_t2, res_t2 = rodar_motor(pid_teste2, verbose=True)
    resumo(pid_teste2, res_t2)
    print('\nÁrvore por_que():')
    print(json.dumps(eng_t2.por_que(res_t2['decisao']), indent=2, ensure_ascii=False))
else:
    print('⚠️ Apenas 1 caso disponível — precisa de pelo menos 2 imagens.')


  CASO bed_not_stick_1 | Fatos iniciais: ['anomalia_pca(detectada)', 'confianca(baixa)', 'criticidade(alta)', 'evento_visao(bed_no_stick)', 'prazo(longo)', 'vibracao(normal)']
  Ciclo 1: FIRE 'R19_PCA_DRIFT_DETECTADO' → 'drift_detectado(sim)'
  Ciclo 2: FIRE 'R20_INVESTIGAR_DRIFT_SVM_NORMAL_CRITICO' → 'decisao(INVESTIGAR)'
  Ciclo 3: FIRE 'R03_VIBRACAO_NORMAL_RISCO_BAIXO' → 'risco_mecanico(baixo)'
  Ciclo 4: agenda vazia — motor para.

--- RESUMO bed_not_stick_1 ---
  Azure   : bed_no_stick (prob=0.44)
  SVM     : no_defected
  PCA     : anomalia=True, mse=1.47701
  Decisão : decisao(INVESTIGAR)
  WM final: ['anomalia_pca(detectada)', 'confianca(baixa)', 'criticidade(alta)', 'decisao(INVESTIGAR)', 'drift_detectado(sim)', 'evento_visao(bed_no_stick)', 'prazo(longo)', 'risco_mecanico(baixo)', 'vibracao(normal)']

Árvore por_que():
{
  "fato": "decisao(INVESTIGAR)",
  "regra": "R20_INVESTIGAR_DRIFT_SVM_NORMAL_CRITICO",
  "justificativa": "PCA detectou drift mas SVM diz normal — alerta pr

---
## 8.1 Prévia — Score de Risco com Dados Reais

A célula abaixo aplica a lógica de decisão gradual por score diretamente nos fatos reais
que acabamos de construir.

In [44]:
# ============================================================
# SCORE DE RISCO — mesma lógica da Aula 5, agora com dados reais
# ============================================================

# Tabela de pesos: cada fato simbólico tem um valor de risco associado.
# Quanto maior o peso, mais aquele sinal contribui para uma decisão grave.
# Fatos que não estão aqui valem 0 — ex: no_defected, vibracao(normal).
# Esses pesos são uma decisão de engenharia — no projeto final
# vocês vão definir e justificar os valores para todas as classes.

PESOS_SCORE = {
    # visao (api azure)
    'evento_visao(leg_broken)':   45,  # fratura estrutural — risco mais alto
    'evento_visao(no_support)':   35,  # suporte ausente
    'evento_visao(bed_no_stick)': 30,  # problema de adesão à base
    'evento_visao(no_bottom)':    40,  # fundo da peça ausente

    # vibração (modelos supervisionados — SVM)
    'vibracao(erro)':             40,  # anomalia mecânica confirmada pelo SVM

    # PCA — detecção de drift (Aula 8)
    'anomalia_pca(detectada)':    30,  # PCA detectou padrão fora da distribuição normal
                                       # Peso 30: menos que vibracao(erro) porque drift é
                                       # um alerta precoce, não uma falha confirmada.

    # criticidade (arquivo contexto)
    'criticidade(alta)':          20,  # peça importante — impacto alto se falhar
    'criticidade(media)':          8,  # peça moderada — impacto menor

    # prazo (arquivo contexto)
    'prazo(curto)':               10,  # urgência operacional — entrega próxima
}


def calcular_score(fatos, confianca_num):

    # Filtra apenas os fatos que têm peso definido no PESOS_SCORE.
    # Fatos sem peso — como vibracao(normal) e no_defected — são ignorados,
    # pois representam o estado esperado e não contribuem para o risco.
    # O resultado é um dicionário: { fato: peso } para rastreabilidade.
    # Registra quais fatos contribuíram e com qual peso
    contribuicoes = {f: PESOS_SCORE[f] for f in fatos if f in PESOS_SCORE}

    # Soma todos os pesos dos fatos que contribuíram.
    # Cada fato presente na WM adiciona seu peso ao risco total.
    score = sum(contribuicoes.values())


    # Teto de 100: garante que o score não ultrapasse o limite máximo
    # mesmo quando muitos sinais graves ocorrem simultaneamente.
    score = min(score, 100)


    # Converte o score numérico em ação — mesma tabela de faixas da Aula 5.
    # O score não decide sozinho: ele é o resumo quantitativo do risco.
    # A ação é a resposta proporcional a esse risco.
    if   score < 30: acao = 'GO'         # risco baixo — seguir normalmente
    elif score < 50: acao = 'MONITORAR'  # risco moderado — acompanhar de perto
    elif score < 70: acao = 'AJUSTAR'    # risco alto — corrigir parâmetros
    elif score < 85: acao = 'INVESTIGAR' # risco crítico — investigar antes de decidir
    else:            acao = 'NOGO'       # risco máximo — parar imediatamente

    # Monta a justificativa mostrando cada fato que contribuiu e seu peso.
    # É o equivalente ao por_que() do motor — só que para o score.
    # Exemplo: criticidade(alta)(+20) | anomalia_pca(detectada)(+30) | prazo(curto)(+10) = 60 → AJUSTAR
    justificativa = ' | '.join(f'{f}(+{p})' for f, p in contribuicoes.items())
    justificativa += f' = {score} → {acao}'

    # Retorna os três valores:
    # score       → o número calculado (para exibir na tabela)
    # acao        → a decisão proporcional ao risco (para exibir na tabela)
    # justificativa → a cadeia de raciocínio auditável (equivalente ao por_que())
    return score, acao, justificativa


# --- Demonstração com o primeiro caso disponível ---

if FATOS_REAIS:
    pid = list(FATOS_REAIS.keys())[0]
    fatos = list(FATOS_REAIS[pid])
    prob  = OUTPUTS_BRUTOS[pid]['azure']['probabilidade']

    score, acao, justificativa = calcular_score(fatos, confianca_num=prob)

    print(f'Demo — {pid}:')
    print(f'  Fatos    : {sorted(fatos)}')
    print(f'  Score    : {score}')
    print(f'  Decisão  : {acao}')
    print(f'  Justificativa : {justificativa}')
else:
    print('⚠️ Nenhum fato real disponível — execute as células anteriores primeiro.')

Demo — bed_not_stick_0:
  Fatos    : ['anomalia_pca(detectada)', 'confianca(alta)', 'criticidade(alta)', 'evento_visao(bed_no_stick)', 'prazo(longo)', 'vibracao(normal)']
  Score    : 80
  Decisão  : INVESTIGAR
  Justificativa : evento_visao(bed_no_stick)(+30) | criticidade(alta)(+20) | anomalia_pca(detectada)(+30) = 80 → INVESTIGAR


---
## 9. Tabela Comparativa — Todos os Casos

In [45]:
linhas = []
for item in IMAGENS:
    pid = item['id']
    _, res = rodar_motor(pid, verbose=False)
    out = OUTPUTS_BRUTOS[pid]
    ctx = df_contexto.loc[pid].to_dict()
    fatos = list(FATOS_REAIS[pid])
    prob  = OUTPUTS_BRUTOS[pid]['azure']['probabilidade']

    # PCA info
    pca_info = out.get('pca', {})
    pca_str  = '🔴 drift' if pca_info.get('anomalia', False) else '🟢 ok'

    # adicionamos o calculo do score
    score, acao_score, justificativa = calcular_score(fatos, confianca_num=prob)

    linhas.append({
        'ID':               pid,
        'Azure (tag)':      out['azure']['tag'],
        'Confiança':        'alta' if prob >= LIMITES['confianca_alta'] else 'baixa',
        'SVM':              out['svm']['classe'],
        'PCA':              pca_str,
        'Criticidade':      ctx['criticidade'],
        'Decisão (Motor)':  res['decisao'],

        # acrescentamos 2 colunas (decisao por score e justificativa)
        'Score':            score,
        'Decisão (Score)':  f'acao({acao_score})',
        'Justificativa':    justificativa,
    })

df = pd.DataFrame(linhas)

def colorir_decisao(val):
    cores = {
        'decisao(GO)':         'background-color: #2ecc71; color: white; font-weight: bold',
        'acao(GO)':            'background-color: #2ecc71; color: white; font-weight: bold',
        'decisao(MONITORAR)':  'background-color: #a8e6a3; color: black; font-weight: bold',
        'acao(MONITORAR)':     'background-color: #a8e6a3; color: black; font-weight: bold',
        'decisao(AJUSTAR)':    'background-color: #f39c12; color: white; font-weight: bold',
        'acao(AJUSTAR)':       'background-color: #f39c12; color: white; font-weight: bold',
        'decisao(INVESTIGAR)': 'background-color: #e67e22; color: white; font-weight: bold',
        'acao(INVESTIGAR)':    'background-color: #e67e22; color: white; font-weight: bold',
        'decisao(NOGO)':       'background-color: #e74c3c; color: white; font-weight: bold',
        'acao(NOGO)':          'background-color: #e74c3c; color: white; font-weight: bold',
        'decisao(INDEFINIDA)': 'background-color: #95a5a6; color: white; font-weight: bold',
    }
    return cores.get(val, '')

display(
    df.style
    .applymap(colorir_decisao, subset=['Decisão (Motor)', 'Decisão (Score)'])
    .set_caption('Projeto Final — Motor (21 regras) vs Score (com PCA)')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-size', '13px'), ('padding', '8px 12px')]},
        {'selector': 'td', 'props': [('padding', '8px 12px'), ('font-size', '13px'), ('border-bottom', '1px solid #ddd')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]},
    ])
    .hide(axis='index')
)

C:\Users\44057824820\AppData\Local\Temp\ipykernel_20128\2459742931.py:52: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(colorir_decisao, subset=['Decisão (Motor)', 'Decisão (Score)'])


ID,Azure (tag),Confiança,SVM,PCA,Criticidade,Decisão (Motor),Score,Decisão (Score),Justificativa
bed_not_stick_0,bed_no_stick,alta,no_defected,🔴 drift,alta,decisao(NOGO),80,acao(INVESTIGAR),evento_visao(bed_no_stick)(+30) | criticidade(alta)(+20) | anomalia_pca(detectada)(+30) = 80 → INVESTIGAR
bed_not_stick_1,bed_no_stick,baixa,no_defected,🔴 drift,alta,decisao(INVESTIGAR),80,acao(INVESTIGAR),evento_visao(bed_no_stick)(+30) | criticidade(alta)(+20) | anomalia_pca(detectada)(+30) = 80 → INVESTIGAR
bed_not_stick_10,bed_no_stick,baixa,no_defected,🔴 drift,baixa,decisao(MONITORAR),60,acao(AJUSTAR),evento_visao(bed_no_stick)(+30) | anomalia_pca(detectada)(+30) = 60 → AJUSTAR
leg_broken_118,no_bottom,baixa,no_defected,🔴 drift,baixa,decisao(MONITORAR),70,acao(INVESTIGAR),evento_visao(no_bottom)(+40) | anomalia_pca(detectada)(+30) = 70 → INVESTIGAR
leg_broken_119,no_bottom,baixa,no_defected,🔴 drift,alta,decisao(INVESTIGAR),90,acao(NOGO),evento_visao(no_bottom)(+40) | criticidade(alta)(+20) | anomalia_pca(detectada)(+30) = 90 → NOGO
leg_broken_120,leg_broken,baixa,no_defected,🔴 drift,media,decisao(INDEFINIDA),83,acao(INVESTIGAR),evento_visao(leg_broken)(+45) | criticidade(media)(+8) | anomalia_pca(detectada)(+30) = 83 → INVESTIGAR
no_bottom_4,no_bottom,baixa,no_defected,🔴 drift,baixa,decisao(MONITORAR),70,acao(INVESTIGAR),evento_visao(no_bottom)(+40) | anomalia_pca(detectada)(+30) = 70 → INVESTIGAR
no_bottom_5,no_bottom,baixa,no_defected,🔴 drift,media,decisao(INDEFINIDA),88,acao(NOGO),evento_visao(no_bottom)(+40) | criticidade(media)(+8) | prazo(curto)(+10) | anomalia_pca(detectada)(+30) = 88 → NOGO
no_bottom_6,no_bottom,baixa,no_defected,🔴 drift,baixa,decisao(MONITORAR),70,acao(INVESTIGAR),evento_visao(no_bottom)(+40) | anomalia_pca(detectada)(+30) = 70 → INVESTIGAR
no_support_234,no_support,alta,no_defected,🟢 ok,baixa,decisao(INDEFINIDA),35,acao(MONITORAR),evento_visao(no_support)(+35) = 35 → MONITORAR


---
## 9.1 Entendendo as duas colunas da tabela

A tabela acima mostra duas formas diferentes de chegar a uma decisão
para a mesma peça. É importante entender o que cada coluna representa
e por que elas podem divergir.

---

**Coluna — Decisão (Aula 6)**

O motor percorre a `BASE_REGRAS` e dispara as regras cujas condições
estão na WorkingMemory.

O motor só enxerga o que as regras cobrem. Se nenhuma regra se aplica,
a decisão é `INDEFINIDA`. Se a peça tem `no_support` com alta confiança,
por exemplo, o motor da Aula 6 não tem resposta — a regra não existe ainda.

---

**Coluna — Decisão (Score)**

O score percorre os mesmos fatos e soma os pesos de cada sinal.

O score sempre produz uma decisão — nunca fica `INDEFINIDO` — porque
qualquer combinação de fatos resulta em um número que cai em alguma faixa.
Mas ele não explica o raciocínio passo a passo como o `por_que()` faz.

---

**Por que as duas podem divergir?**

O motor decide pela **lógica das regras** — ele precisa que os fatos
se encaixem exatamente nas condições escritas pelo especialista.

O score decide pela **matemática dos pesos** — ele pondera o conjunto
todo de sinais, incluindo os que não disparariam nenhuma regra sozinhos.

**O que muda no projeto final**

Hoje o score e o motor rodam separados e produzem decisões independentes.

No projeto final vocês vão uni-los: o score gera um predicado simbólico
(`score_risco(MONITORAR)`) que entra na WorkingMemory como um fato,
e as regras graduais usam esse fato para decidir entre as 5 ações.
Para isso, vocês precisarão criar regras como:

    SE  score_risco(MONITORAR)
    →   decisao(MONITORAR)   [prioridade 30]
    
    SE  score_risco(AJUSTAR)
    →   decisao(AJUSTAR)     [prioridade 50]

Sem essas regras, o predicado entra na WM mas ninguém o usa —
e o motor continua decidindo como se o score não existisse.
Criar essas regras, justificar os pesos e cobrir todas as classes
do Azure é exatamente o trabalho do projeto final.



---
## 11. Reflexão do Grupo

> **Instruções:** cada integrante preenche a sua seção respondendo às três perguntas abaixo. Seja direto — o objetivo é mostrar que o grupo pensou nas decisões, não apenas copiou código.

---

### Integrante 1 — Luiz Nunes

**Quais foram as decisões de engenharia mais difíceis?**

A decisão mais difícil foi definir os limiares de grounding — em especial o corte de `confianca_alta = 0.80`. Esse valor não saiu dos dados; ele reflete uma tolerância ao risco. Se o modelo acerta 79% das vezes, o sistema o trata como "baixa confiança" e pode travar uma peça que estaria boa. Abaixar para 0.70 reduziria falsos NOGO, mas aumentaria o risco de deixar passar defeitos. Mantivemos 0.80 como posição conservadora para um ambiente industrial onde o custo de um defeito não detectado é maior que o custo de parar a linha.

A segunda decisão difícil foi cobrir todas as combinações de criticidade e prazo sem deixar lacunas de `INDEFINIDA`. A estrutura original tinha apenas 21 regras e deixava casos como `leg_broken + criticidade baixa` sem resposta. Foi necessário adicionar 10 regras (R22–R31) mapeando sistematicamente todas as combinações presentes no `contexto_pecas.csv`.

**Em algum caso o motor gerou `decisao(INDEFINIDA)`? Por quê?**

Sim, antes da expansão das regras. As peças `leg_broken_118` (criticidade baixa) e `leg_broken_120` (criticidade média) retornavam `INDEFINIDA` porque a base original só tratava `leg_broken` com `criticidade(alta)`. O mesmo ocorria para `bed_not_stick_10`, `no_support_234`, `no_support_235` e `no_bottom_5`. O motor não "chuta" — se nenhuma regra cobre a situação, ele retorna `INDEFINIDA` honestamente. Isso é um recurso, não um bug: força o engenheiro a ser explícito sobre cada cenário.

**O que você mudaria com mais dados ou mais tempo?**

Com mais dados, calibraria os pesos do score usando análise de falha real (FMEA) em vez de estimativas. O peso de `leg_broken` (45) foi definido por julgamento — com histórico de falhas da linha, seria possível calcular o custo médio de cada tipo de defeito e derivar os pesos matematicamente.

Com mais tempo, integraria o score ao motor de forma que `score_risco(X)` entrasse na WorkingMemory como fato, permitindo que as regras graduais consumissem a decisão ponderada e eliminassem a coluna paralela na tabela comparativa.

---

### Integrante 2 — *(preencher)*

**Quais foram as decisões de engenharia mais difíceis?**

*(sua resposta aqui)*

**Em algum caso o motor gerou `decisao(INDEFINIDA)`? Por quê?**

*(sua resposta aqui)*

**O que você mudaria com mais dados ou mais tempo?**

*(sua resposta aqui)*

---

### Integrante 3 — *(preencher)*

**Quais foram as decisões de engenharia mais difíceis?**

*(sua resposta aqui)*

**Em algum caso o motor gerou `decisao(INDEFINIDA)`? Por quê?**

*(sua resposta aqui)*

**O que você mudaria com mais dados ou mais tempo?**

*(sua resposta aqui)*

---

### Integrante 4 — *(preencher)*

**Quais foram as decisões de engenharia mais difíceis?**

*(sua resposta aqui)*

**Em algum caso o motor gerou `decisao(INDEFINIDA)`? Por quê?**

*(sua resposta aqui)*

**O que você mudaria com mais dados ou mais tempo?**

*(sua resposta aqui)*
